In [2]:
"""
α_int / α_ext 计算 + Pattern Quality 筛选
================================================
α_int(u,k) = e_u^T v_k          （用户对 pattern 的内部响应）
α_ext(u,k) = |B_u ∩ I_k| / |B_u| （用户对 pattern 的外部行为强度）

Quality 筛选：
  Pearson  相关系数 ≥ δ=0.5
  Spearman 相关系数 ≥ δ=0.5
  分别筛选，对比两种方式的结果

保存至: SAVE_DIR/
  alpha_int.pt          — α_int 矩阵 (n_users, K)
  alpha_ext.pt          — α_ext 矩阵 (n_users, K)
  quality_scores.pt     — 每个 pattern 的 Pearson 与 Spearman 系数
  high_quality_pearson.pt  — Pearson 筛选通过的 pattern 集合 H
  high_quality_spearman.pt — Spearman 筛选通过的 pattern 集合 H
================================================
"""

import os
import torch
import numpy as np
from scipy.stats import pearsonr, spearmanr

# ─────────────────────────────────────────────────────────────
# 路径
# ─────────────────────────────────────────────────────────────
EMBED_PATH       = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/模型文件/embeddings_best.pt"
BASIS_PATH       = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_basis/pattern_basis.pt"
ITEM_SETS_PATH   = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_basis/pattern_item_sets.pt"
DATA_PATH        = "/Users/yubinhao/ml-1m/Amazon Review/数据集/Amazon_Reviews_Processed"
SAVE_DIR         = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/α与quality"

DELTA            = 0.5     # 质量筛选阈值
MIN_INTERACTIONS = 5       # 与训练时一致


# ─────────────────────────────────────────────────────────────
# 加载数据
# ─────────────────────────────────────────────────────────────
print("[加载] embeddings...")
ckpt     = torch.load(EMBED_PATH, map_location="cpu")
user_emb = ckpt["user_embeddings"].numpy()   # (n_users, d)
item_emb = ckpt["item_embeddings"].numpy()   # (n_items, d)
n_users, d = user_emb.shape
n_items    = item_emb.shape[0]
print(f"  user: {user_emb.shape}, item: {item_emb.shape}")

print("[加载] pattern basis...")
basis_ckpt = torch.load(BASIS_PATH, map_location="cpu")
V = basis_ckpt["pattern_basis"].numpy()      # (K, d)
K = V.shape[0]
print(f"  pattern basis: {V.shape}")

print("[加载] pattern item sets...")
sets_ckpt        = torch.load(ITEM_SETS_PATH, map_location="cpu")
pattern_item_sets = sets_ckpt["pattern_item_sets"]   # dict: k → list of item indices
TOP_N = sets_ckpt["top_n"]
print(f"  TOP_N={TOP_N}, K={K}")

print("[加载] 用户交互数据...")
import pandas as pd
df = pd.read_csv(DATA_PATH, usecols=["user_id", "item_id", "rating"])
user_counts = df.groupby("user_id").size()
valid_users = user_counts[user_counts >= MIN_INTERACTIONS].index
df = df[df["user_id"].isin(valid_users)].copy()

# 重建与训练时一致的 user2idx / item2idx
user_ids = sorted(df["user_id"].unique())
item_ids = sorted(df["item_id"].unique())
user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {it: i for i, it in enumerate(item_ids)}
df["uid"] = df["user_id"].map(user2idx)
df["iid"] = df["item_id"].map(item2idx)

# 构造 B_u：每个用户的 item index 集合
print("  构造 B_u（用户交互 item 集合）...")
user_item_sets = df.groupby("uid")["iid"].apply(set).to_dict()
print(f"  共 {len(user_item_sets)} 个用户")


# ─────────────────────────────────────────────────────────────
# 计算 α_int(u,k) = e_u^T v_k
# ─────────────────────────────────────────────────────────────
print("\n[α_int] 计算内部响应矩阵...")
# 矩阵乘法：(n_users, d) @ (d, K) → (n_users, K)
alpha_int = user_emb @ V.T    # (n_users, K)
print(f"  α_int shape: {alpha_int.shape}")
print(f"  α_int 统计: mean={alpha_int.mean():.4f}, std={alpha_int.std():.4f}, "
      f"min={alpha_int.min():.4f}, max={alpha_int.max():.4f}")


# ─────────────────────────────────────────────────────────────
# 计算 α_ext(u,k) = |B_u ∩ I_k| / |B_u|
# ─────────────────────────────────────────────────────────────
print("\n[α_ext] 计算外部行为强度矩阵...")
alpha_ext = np.zeros((n_users, K), dtype=np.float32)

for k in range(K):
    I_k = set(pattern_item_sets[k])   # pattern k 的代表 item 集合
    for uid in range(n_users):
        B_u = user_item_sets.get(uid, set())
        if len(B_u) == 0:
            alpha_ext[uid, k] = 0.0
        else:
            alpha_ext[uid, k] = len(B_u & I_k) / len(B_u)

    if (k + 1) % 5 == 0 or k == K - 1:
        print(f"  pattern {k+1}/{K} 完成，"
              f"α_ext[:,{k}] mean={alpha_ext[:, k].mean():.4f}, "
              f"nonzero={np.count_nonzero(alpha_ext[:, k])}")

print(f"\n  α_ext shape: {alpha_ext.shape}")
print(f"  α_ext 统计: mean={alpha_ext.mean():.4f}, std={alpha_ext.std():.4f}, "
      f"min={alpha_ext.min():.4f}, max={alpha_ext.max():.4f}")


# ─────────────────────────────────────────────────────────────
# Pattern Quality：Pearson 与 Spearman
# ─────────────────────────────────────────────────────────────
print("\n[Quality] 计算 Pearson 与 Spearman 相关系数...")

pearson_scores  = np.zeros(K)
spearman_scores = np.zeros(K)

for k in range(K):
    a_int = alpha_int[:, k]
    a_ext = alpha_ext[:, k]

    # Pearson
    r_p, _ = pearsonr(a_int, a_ext)
    pearson_scores[k] = r_p if not np.isnan(r_p) else 0.0

    # Spearman
    r_s, _ = spearmanr(a_int, a_ext)
    spearman_scores[k] = r_s if not np.isnan(r_s) else 0.0

# 打印每个 pattern 的分数
print(f"\n  {'Pattern':>8} {'Pearson':>10} {'Spearman':>10} {'Pearson≥0.5':>12} {'Spearman≥0.5':>13}")
print("  " + "-" * 58)
for k in range(K):
    p_pass = "✓" if pearson_scores[k]  >= DELTA else "✗"
    s_pass = "✓" if spearman_scores[k] >= DELTA else "✗"
    print(f"  {k:>8d} {pearson_scores[k]:>10.4f} {spearman_scores[k]:>10.4f} "
          f"{'':>4}{p_pass:>6}{'':>6}{s_pass:>6}")


# ─────────────────────────────────────────────────────────────
# 高质量 Pattern 集合 H
# ─────────────────────────────────────────────────────────────
H_pearson  = [k for k in range(K) if pearson_scores[k]  >= DELTA]
H_spearman = [k for k in range(K) if spearman_scores[k] >= DELTA]
H_both     = [k for k in range(K) if pearson_scores[k] >= DELTA and spearman_scores[k] >= DELTA]
H_either   = [k for k in range(K) if pearson_scores[k] >= DELTA or  spearman_scores[k] >= DELTA]

print(f"\n[筛选结果] δ={DELTA}")
print(f"  Pearson  通过: {len(H_pearson):>3} 个 → {H_pearson}")
print(f"  Spearman 通过: {len(H_spearman):>3} 个 → {H_spearman}")
print(f"  两者均通过:     {len(H_both):>3} 个 → {H_both}")
print(f"  任一通过:       {len(H_either):>3} 个 → {H_either}")


# ─────────────────────────────────────────────────────────────
# 保存
# ─────────────────────────────────────────────────────────────
os.makedirs(SAVE_DIR, exist_ok=True)

# α_int 矩阵
torch.save(
    {"alpha_int": torch.FloatTensor(alpha_int),   # (n_users, K)
     "K": K, "n_users": n_users},
    os.path.join(SAVE_DIR, "alpha_int.pt")
)

# α_ext 矩阵
torch.save(
    {"alpha_ext": torch.FloatTensor(alpha_ext),   # (n_users, K)
     "K": K, "n_users": n_users, "top_n": TOP_N},
    os.path.join(SAVE_DIR, "alpha_ext.pt")
)

# Quality scores
torch.save(
    {"pearson_scores":  pearson_scores.tolist(),
     "spearman_scores": spearman_scores.tolist(),
     "delta": DELTA, "K": K},
    os.path.join(SAVE_DIR, "quality_scores.pt")
)

# 高质量集合
torch.save(
    {"H": H_pearson, "delta": DELTA, "method": "pearson"},
    os.path.join(SAVE_DIR, "high_quality_pearson.pt")
)
torch.save(
    {"H": H_spearman, "delta": DELTA, "method": "spearman"},
    os.path.join(SAVE_DIR, "high_quality_spearman.pt")
)

print(f"\n[完成] 保存至 {SAVE_DIR}")
print(f"  alpha_int.pt             — α_int 矩阵 {alpha_int.shape}")
print(f"  alpha_ext.pt             — α_ext 矩阵 {alpha_ext.shape}")
print(f"  quality_scores.pt        — Pearson & Spearman 系数")
print(f"  high_quality_pearson.pt  — H_pearson  ({len(H_pearson)} patterns)")
print(f"  high_quality_spearman.pt — H_spearman ({len(H_spearman)} patterns)")

[加载] embeddings...
  user: (27794, 64), item: (144853, 64)
[加载] pattern basis...
  pattern basis: (20, 64)
[加载] pattern item sets...
  TOP_N=2000, K=20
[加载] 用户交互数据...
  构造 B_u（用户交互 item 集合）...
  共 27794 个用户

[α_int] 计算内部响应矩阵...
  α_int shape: (27794, 20)
  α_int 统计: mean=0.0821, std=0.6173, min=-5.7272, max=22.9522

[α_ext] 计算外部行为强度矩阵...
  pattern 5/20 完成，α_ext[:,4] mean=0.1688, nonzero=19846
  pattern 10/20 完成，α_ext[:,9] mean=0.0420, nonzero=7945
  pattern 15/20 完成，α_ext[:,14] mean=0.0410, nonzero=7379
  pattern 20/20 完成，α_ext[:,19] mean=0.0294, nonzero=5860

  α_ext shape: (27794, 20)
  α_ext 统计: mean=0.0762, std=0.1334, min=0.0000, max=1.0000

[Quality] 计算 Pearson 与 Spearman 相关系数...

   Pattern    Pearson   Spearman  Pearson≥0.5  Spearman≥0.5
  ----------------------------------------------------------
         0     0.3748     0.3306          ✗           ✗
         1     0.5033     0.5419          ✓           ✓
         2     0.5001     0.5419          ✓           ✓
         3     